# Importing Library

In [1]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.layers import Input, BatchNormalization, Dropout, Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import SparseCategoricalAccuracy, Precision, Recall, AUC
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import math
import json

2025-09-05 17:23:13.724802: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757071393.856849     735 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757071393.891431     735 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1757071394.164317     735 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1757071394.164390     735 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1757071394.164392     735 computation_placer.cc:177] computation placer alr

# Importing Dataset

In [2]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 16
SEED = 123
train_ds = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    validation_split=0.2,
    subset="training",
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    validation_split=0.2,
    subset="validation",
    seed=SEED
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    "data/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True
)

Found 9650 files belonging to 3 classes.
Using 7720 files for training.


I0000 00:00:1757071407.801739     735 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5563 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060, pci bus id: 0000:01:00.0, compute capability: 8.9


Found 9650 files belonging to 3 classes.
Using 1930 files for validation.
Found 2278 files belonging to 3 classes.


# Data Preprocessing

## Resizing The Image

In [ ]:
def resize_batch(images, labels):
    images = tf.image.resize(images, IMG_SIZE) 
    images = tf.cast(images, tf.float32) / 255.0
    return images, labels

train_ds = train_ds.map(resize_batch).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(resize_batch).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(resize_batch).prefetch(tf.data.AUTOTUNE)

# Model Creation

In [4]:
def build_model(input_shape=IMG_SIZE + (3,)):
    base_model = tf.keras.applications.EfficientNetV2S(
        include_top=False,
        input_shape=input_shape,
        weights='imagenet'
    )
    base_model.trainable = False

    inputs = Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(3, activation='softmax')(x)
    model = Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=[
            SparseCategoricalAccuracy(name='accuracy'),
        ]
    )
    return model

model = build_model()
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-s (Functional)   │ (None, 4, 4, 1280)     │    20,331,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,501,347 (78.21 MB)

 Trainable params: 167,171 (653.01 KB)

 Non-trainable params: 20,334,176 (77.57 MB)

# Training The Model

In [5]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    verbose=1,
)

Epoch 1/10


I0000 00:00:1757071432.241044     969 service.cc:152] XLA service 0x7f0350002720 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1757071432.241072     969 service.cc:160]   StreamExecutor device (0): NVIDIA GeForce RTX 4060, Compute Capability 8.9
2025-09-05 17:23:52.794829: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1757071435.031951     969 cuda_dnn.cc:529] Loaded cuDNN version 90300


  1/483 ━━━━━━━━━━━━━━━━━━━━ 4:24:04 33s/step - accuracy: 0.3125 - loss: 1.7213

I0000 00:00:1757071453.519687     969 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


483/483 ━━━━━━━━━━━━━━━━━━━━ 74s 84ms/step - accuracy: 0.3929 - loss: 1.2465 - val_accuracy: 0.4648 - val_loss: 1.0343
Epoch 2/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 12s 24ms/step - accuracy: 0.4391 - loss: 1.0907 - val_accuracy: 0.4575 - val_loss: 1.0272
Epoch 3/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 13s 27ms/step - accuracy: 0.4487 - loss: 1.0634 - val_accuracy: 0.4596 - val_loss: 1.0249
Epoch 4/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - accuracy: 0.4560 - loss: 1.0465 - val_accuracy: 0.4813 - val_loss: 1.0099
Epoch 5/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - accuracy: 0.4633 - loss: 1.0376 - val_accuracy: 0.4870 - val_loss: 1.0067
Epoch 6/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 13s 27ms/step - accuracy: 0.4667 - loss: 1.0374 - val_accuracy: 0.5047 - val_loss: 1.0025
Epoch 7/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - accuracy: 0.4701 - loss: 1.0308 - val_accuracy: 0.4907 - val_loss: 1.0017
Epoch 8/10
483/483 ━━━━━━━━━━━━━━━━━━━━ 10s 21ms/step - accuracy: 0.4802 - loss: 1.0263 - val_accurac